## Data cleaning from raw data

De las tablas origen se han agrupado por `spred_sheet`, `emision`, `comision`, `siniestros`en un rango de $2021 - 2024$

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")

In [2]:
emision    = pd.read_csv(PROCESSED / "emision_total.csv", dtype=str)
siniestros = pd.read_csv(PROCESSED / "siniestros_total.csv", dtype=str)
comisiones = pd.read_csv(PROCESSED / "comisiones_total.csv", dtype=str)

print(emision.shape, siniestros.shape, comisiones.shape)

(3508593, 12) (297505, 13) (827907, 16)


In [5]:
# Columnas reales de cada tabla (referencia)
SHEETS = {
"emision": emision,
"siniestros": siniestros,
"comisiones": comisiones,
}

clean = {}

for name, df in SHEETS.items():
    header_cols = df.columns.tolist()
    mascara = df.isin(header_cols).any(axis=1)
    df_clean = df[~mascara].reset_index(drop=True)
    clean[name] = df_clean
    print(f"{name}: {len(df):,} → {len(df_clean):,} filas  ({mascara.sum():,} eliminadas)")

emision_c    = clean["emision"]
siniestros_c = clean["siniestros"]
comisiones_c = clean["comisiones"]

emision: 3,508,593 → 3,508,593 filas  (0 eliminadas)
siniestros: 297,505 → 297,505 filas  (0 eliminadas)
comisiones: 827,907 → 827,907 filas  (0 eliminadas)


In [10]:
# Inspección inicial: tipos, nulos reales (NaN) y vacíos ("") por tabla.
# Como se cargó con dtype=str, todos los dtypes serán object — lo relevante
# es detectar strings vacíos que CNSF usa en lugar de NaN, y ver los
# nombres reales de columnas antes de decidir casteos y validaciones.
for name, df in clean.items():
    print(f"\n{'='*50}")
    print(f" {name}  {df.shape}")
    print(f"{'='*50}")
    print(df.dtypes)
    print("\nNulos:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.any() else "  ninguno")
    print("\nVacíos (string ''):")
    vacios = (df == "").sum()
    print(vacios[vacios > 0] if vacios.any() else "  ninguno")
    print("\nMuestra:")
    display(df.head(3))


 emision  (3508593, 12)
ANIO                           str
Indice                         str
Unnamed: 1                     str
EMISIÓN\n(Pesos corrientes)    str
Unnamed: 3                     str
Unnamed: 4                     str
Unnamed: 5                     str
Unnamed: 6                     str
Unnamed: 7                     str
Unnamed: 8                     str
Unnamed: 9                     str
Unnamed: 10                    str
dtype: object

Nulos:
Indice    11
dtype: int64

Vacíos (string ''):
  ninguno

Muestra:


,ANIO,Indice,Unnamed: 1,EMISIÓN\n(Pesos corrientes),Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,2021,EDAD,COBERTURA,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,MONEDA,ENTIDAD,SEXO,FORMA DE VENTA,NUMERO DE ASEGURADOS,PRIMA EMITIDA,SUMA ASEGURADA
1,2021,22,Exención de pago de prima,Vitalicio,Tradicional,Extranjera,Chihuahua,Femenino,Agentes Persona Física,1,127.68,0
2,2021,47,Fallecimiento,Vitalicio,Tradicional,Extranjera,Coahuila,Masculino,Fuerza de Venta Interna o Casa Matriz,6,146947.26,9432861.69



 siniestros  (297505, 13)
ANIO                              str
Indice                            str
SINIESTROS\n(Pesos corrientes)    str
Unnamed: 2                        str
Unnamed: 3                        str
Unnamed: 4                        str
Unnamed: 5                        str
Unnamed: 6                        str
Unnamed: 7                        str
Unnamed: 8                        str
Unnamed: 9                        str
Unnamed: 10                       str
Unnamed: 11                       str
dtype: object

Nulos:
Unnamed: 4    1
dtype: int64

Vacíos (string ''):
  ninguno

Muestra:


,ANIO,Indice,SINIESTROS\n(Pesos corrientes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,2021,EDAD,COBERTURA,ENTIDAD,CAUSA DEL SINIESTRO,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,SEXO,NUMERO DE SINIESTROS,MONTO RECLAMADO,VENCIMIENTOS,MONTO PAGADO,MONTO DE REASEGURO
1,2021,61,Invalidez total y permanente,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Flexible sin tasa garantizada,Femenino,1,1485500.71,0,1489939.28,0
2,2021,46,Asistencias,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-5800.39,0,0,0



 comisiones  (827907, 16)
ANIO                              str
Indice                            str
COMISIONES\n(Pesos corrientes)    str
Unnamed: 2                        str
Unnamed: 3                        str
Unnamed: 4                        str
Unnamed: 5                        str
Unnamed: 6                        str
Unnamed: 7                        str
Unnamed: 8                        str
Unnamed: 9                        str
Unnamed: 10                       str
Unnamed: 11                       str
Unnamed: 12                       str
Unnamed: 13                       str
Unnamed: 14                       str
dtype: object

Nulos:
  ninguno

Vacíos (string ''):
  ninguno

Muestra:


,ANIO,Indice,COMISIONES\n(Pesos corrientes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,2021,EDAD,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,MONEDA,ENTIDAD,SEXO,FORMA DE VENTA,TIPO DIVIDENDO,NUMERO DE ASEGURADOS,PRIMA CEDIDA,COMISIONES DIRECTAS,FONDO DE INVERSIÓN,FONDO DE ADMINISTRACION,MONTO DE DIVIDENDOS,MONTO DE RESCATE
1,2021,0,Temporal,Tradicional,Nacional,Morelos,Masculino,Agentes Persona Física,Sin dividendo,1,0,275,0,0,0,0
2,2021,0,Temporal,Tradicional,Nacional,Veracruz,Femenino,Agentes Persona Física,Sin dividendo,1,0,192,0,0,0,0


In [9]:
# La fila 0 de cada tabla contiene los encabezados reales.
# Se promueve como header y se elimina de los datos.
fixed = {}

for name, df in clean.items():
    # Tomar fila 0 como nuevos nombres de columna
    nuevos_headers = df.iloc[0].tolist()
    df_fixed = df.iloc[1:].copy()
    df_fixed.columns = nuevos_headers
    df_fixed = df_fixed.reset_index(drop=True)
    fixed[name] = df_fixed
    print(f"{name}: columnas → {nuevos_headers}")

emision_f    = fixed["emision"]
siniestros_f = fixed["siniestros"]
comisiones_f = fixed["comisiones"]

emision: columnas → ['2021', 'EDAD', 'COBERTURA', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'MONEDA', 'ENTIDAD ', 'SEXO', 'FORMA DE VENTA', 'NUMERO DE ASEGURADOS', 'PRIMA EMITIDA', 'SUMA ASEGURADA']
siniestros: columnas → ['2021', 'EDAD', 'COBERTURA', 'ENTIDAD', 'CAUSA DEL SINIESTRO', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'SEXO', 'NUMERO DE SINIESTROS', 'MONTO RECLAMADO', 'VENCIMIENTOS', 'MONTO PAGADO', 'MONTO DE REASEGURO']
comisiones: columnas → ['2021', 'EDAD', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'MONEDA', 'ENTIDAD ', 'SEXO', 'FORMA DE VENTA', 'TIPO DIVIDENDO', 'NUMERO DE ASEGURADOS', 'PRIMA CEDIDA', 'COMISIONES DIRECTAS', 'FONDO DE INVERSIÓN', 'FONDO DE ADMINISTRACION', 'MONTO DE DIVIDENDOS', 'MONTO DE RESCATE']


In [11]:
# Inspección inicial: tipos, nulos reales (NaN) y vacíos ("") por tabla.
# Como se cargó con dtype=str, todos los dtypes serán object — lo relevante
# es detectar strings vacíos que CNSF usa en lugar de NaN, y ver los
# nombres reales de columnas antes de decidir casteos y validaciones.
for name, df in fixed.items():
    print(f"\n{'='*50}")
    print(f" {name}  {df.shape}")
    print(f"{'='*50}")
    print(df.dtypes)
    print("\nNulos:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.any() else "  ninguno")
    print("\nVacíos (string ''):")
    vacios = (df == "").sum()
    print(vacios[vacios > 0] if vacios.any() else "  ninguno")
    print("\nMuestra:")
    display(df.head(3))


 emision  (3508592, 12)
2021                      str
EDAD                      str
COBERTURA                 str
PLAN DE LA POLIZA         str
MODALIDAD DE LA POLIZA    str
MONEDA                    str
ENTIDAD                   str
SEXO                      str
FORMA DE VENTA            str
NUMERO DE ASEGURADOS      str
PRIMA EMITIDA             str
SUMA ASEGURADA            str
dtype: object

Nulos:
EDAD    11
dtype: int64

Vacíos (string ''):
  ninguno

Muestra:


,2021,EDAD,COBERTURA,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,MONEDA,ENTIDAD,SEXO,FORMA DE VENTA,NUMERO DE ASEGURADOS,PRIMA EMITIDA,SUMA ASEGURADA
0,2021,22,Exención de pago de prima,Vitalicio,Tradicional,Extranjera,Chihuahua,Femenino,Agentes Persona Física,1,127.68,0
1,2021,47,Fallecimiento,Vitalicio,Tradicional,Extranjera,Coahuila,Masculino,Fuerza de Venta Interna o Casa Matriz,6,146947.26,9432861.69
2,2021,34,Fallecimiento,Dotal Mixto,Tradicional,Indizada,Querétaro,Femenino,Agentes Persona Física,64,1206118.07,36899161.66



 siniestros  (297504, 13)
2021                      str
EDAD                      str
COBERTURA                 str
ENTIDAD                   str
CAUSA DEL SINIESTRO       str
PLAN DE LA POLIZA         str
MODALIDAD DE LA POLIZA    str
SEXO                      str
NUMERO DE SINIESTROS      str
MONTO RECLAMADO           str
VENCIMIENTOS              str
MONTO PAGADO              str
MONTO DE REASEGURO        str
dtype: object

Nulos:
PLAN DE LA POLIZA    1
dtype: int64

Vacíos (string ''):
  ninguno

Muestra:


,2021,EDAD,COBERTURA,ENTIDAD,CAUSA DEL SINIESTRO,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,SEXO,NUMERO DE SINIESTROS,MONTO RECLAMADO,VENCIMIENTOS,MONTO PAGADO,MONTO DE REASEGURO
0,2021,61,Invalidez total y permanente,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Flexible sin tasa garantizada,Femenino,1,1485500.71,0,1489939.28,0
1,2021,46,Asistencias,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-5800.39,0,0,0
2,2021,46,Fallecimiento,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-469891.6,0,0,0



 comisiones  (827906, 16)
2021                       str
EDAD                       str
PLAN DE LA POLIZA          str
MODALIDAD DE LA POLIZA     str
MONEDA                     str
ENTIDAD                    str
SEXO                       str
FORMA DE VENTA             str
TIPO DIVIDENDO             str
NUMERO DE ASEGURADOS       str
PRIMA CEDIDA               str
COMISIONES DIRECTAS        str
FONDO DE INVERSIÓN         str
FONDO DE ADMINISTRACION    str
MONTO DE DIVIDENDOS        str
MONTO DE RESCATE           str
dtype: object

Nulos:
  ninguno

Vacíos (string ''):
  ninguno

Muestra:


,2021,EDAD,PLAN DE LA POLIZA,MODALIDAD DE LA POLIZA,MONEDA,ENTIDAD,SEXO,FORMA DE VENTA,TIPO DIVIDENDO,NUMERO DE ASEGURADOS,PRIMA CEDIDA,COMISIONES DIRECTAS,FONDO DE INVERSIÓN,FONDO DE ADMINISTRACION,MONTO DE DIVIDENDOS,MONTO DE RESCATE
0,2021,0,Temporal,Tradicional,Nacional,Morelos,Masculino,Agentes Persona Física,Sin dividendo,1,0,275,0,0,0,0
1,2021,0,Temporal,Tradicional,Nacional,Veracruz,Femenino,Agentes Persona Física,Sin dividendo,1,0,192,0,0,0,0
2,2021,0,Temporal,Tradicional,Nacional,Veracruz,Masculino,Agentes Persona Física,Sin dividendo,1,0,192,0,0,0,0


In [ ]:
# Load fixed dataframes into clean dict for export
CLEAN = Path("../data/processed/clean")
CLEAN.mkdir(exist_ok=True)

for name, df in fixed.items():
    out = CLEAN / f"{name}_clean.csv"
    df.to_csv(out, index=False)
    print(f"✓ {out.name}  {df.shape}")

✓ emision_clean.csv  (3508592, 12)
✓ siniestros_clean.csv  (297504, 13)
✓ comisiones_clean.csv  (827906, 16)
